In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

from adjustText import adjust_text

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden_old.h5ad")

data.shape

In [ ]:
data.layers['raw'] = np.array(data.layers['raw'].astype(int).todense())

# Pseudobulking

In [ ]:
import anndata as ad

pseudobulks = []

n_pseduo_bulks = 1

for cell_type in sorted(list(set(data.obs['cell_type']))):
    if cell_type == 'trash':
        continue
    for condition in set(data.obs['exp_condition']):
        for time in sorted(list(set(data.obs['time']))):
            # print(cell_type,condition,time)
            mask = (
                (data.obs['cell_type'] == cell_type) & \
                (data.obs['exp_condition'] == condition) & \
                (data.obs['time'] == time) \
            )
            mask_split = np.random.randint(n_pseduo_bulks,size=mask.shape)
            submasks = [mask & (mask_split == i) for i in range(n_pseduo_bulks)]
            for submask in submasks:
                counts = data.layers['raw'][submask].sum(axis=0)
                annd = ad.AnnData(counts.reshape(1,-1))
                annd.obs['cell_type'] = cell_type
                annd.obs['exp_condition'] = condition
                annd.obs['time'] = time

                annd.obs['cell_type'] = annd.obs['cell_type'].astype('category')
                annd.obs['exp_condition'] = annd.obs['exp_condition'].astype('category')
                annd.obs['time'] = annd.obs['time'].astype('category')
                

                pseudobulks.append(annd)
            
pseudobulks = ad.concat(pseudobulks)
pseudobulks.obs_names_make_unique()
pseudobulks.var_names = data.var_names

In [ ]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=pseudobulks.X+1,
    metadata=pseudobulks.obs,
    # design="~ exp_condition + cell_type + time",
    design="~ exp_condition + cell_type + time + exp_condition:time",
    refit_cooks=True,
    inference=inference,
)

In [ ]:
# Sets RGCs as the reference population for all relative changes

excluded_categories = list(dds.obs["cell_type"].cat.categories.unique())
excluded_categories.remove("RGCs")
dds.obs["cell_type"] = dds.obs["cell_type"].cat.reorder_categories([
    "RGCs",
    *excluded_categories,
]) 
dds.deseq2()
ds = DeseqStats(dds, contrast=["exp_condition", "Mtz", "Cntr"], inference=inference)
ds.summary()

In [ ]:
# 1 > (-np.log10(ds.padj))
# np.sum((-1 * np.log10(ds.padj))>1)
# df = ds.results_df
# df.index = data.var_names
# df.to_csv("./tables/global_deseq.csv")

### Volcano Plot

In [ ]:
def deseq_volcano(
    ds_target,
    dataframe,
    label=None,
    lfc_cutoff=1,pval_cutoff=1,
    padj_cutoff=0.05,
    use_padj=True,
    figsize=(8,12),
    path=None,
):

    genes_to_mark = (
        (np.abs(ds_target.LFC['exp_condition[T.Mtz]']) > lfc_cutoff) &
        (-np.log10(ds_target.p_values) > pval_cutoff)
    )


    plt.figure(figsize=figsize)
    plt.title(f"DESeq2, {label}")

    x = ds_target.LFC['exp_condition[T.Mtz]']
    # y = -np.log10(ds_target.p_values)
    y = -np.log10(ds_target.padj) if use_padj else -np.log10(ds_target.p_values)
    plt.scatter(
        x,y,
        s=1,
    )
    ax = plt.gca()
    offset = 0.05


    texts = []
    for gene, xi, yi in zip(dataframe.var_names[genes_to_mark], x[genes_to_mark], y[genes_to_mark]):
        texts.append(ax.text(xi, yi, gene, fontsize=10))

    # Adjust label positions to avoid overlaps
    adjust_text(
        texts,
        # x=x.values, y=y.values,   # repel away from ALL points, not just labeled ones
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        expand=(1.2, 1.4),
    )

    ylims = ax.get_ylim()
    xlims = ax.get_xlim()
    plt.plot([lfc_cutoff,lfc_cutoff],ylims,'k--')
    plt.plot([-lfc_cutoff,-lfc_cutoff],ylims,'k--')
    if use_padj:
        plt.plot(xlims,[padj_cutoff,padj_cutoff],'k--')
    else:
        plt.plot(xlims,[pval_cutoff,pval_cutoff],'k--')


    plt.xlabel("Log2 Fold Change")
    plt.ylabel("-Log10(p-value)")
    plt.tight_layout()
    if path is not None:
        plt.savefig(path,dpi=300)
    plt.show()


In [ ]:
ds.LFC.head()

In [ ]:
# interaction term analysis

In [ ]:
deseq_volcano(
    ds,
    data,
    label="Global, Injury vs Control",
    lfc_cutoff=.5,pval_cutoff=1,
    padj_cutoff=1,
    figsize=(9,7),
    path="./figures/deseq_volcano_global.png"
)

In [ ]:
ds.summary()

In [ ]:
diff_sort_full = np.argsort(ds.LFC['exp_condition[T.Mtz]'])

delta_neg_full = list(zip(data.var_names[diff_sort_full],ds.LFC['exp_condition[T.Mtz]'][diff_sort_full]))[:100]
delta_pos_full = list(zip(data.var_names[diff_sort_full],ds.LFC['exp_condition[T.Mtz]'][diff_sort_full]))[-100:]


In [ ]:
# print("\n".join([f"{str(g).ljust(20)}:\t\t{f:.2f}" for g,f in delta_neg_full]))
print("\n".join([f"{str(g)}" for g,f in delta_neg_full]),file=open("delta_neg_bare_global.txt","w"))
print("\n".join([f"{str(g)}" for g,f in delta_pos_full]),file=open("delta_pos_bare_global.txt","w"))

In [ ]:
print("\n".join([f"{str(g).ljust(20)}:\t\t{f:.2f}" for g,f in delta_pos_full]))

# Subset RGCs

In [ ]:
pseudobulk_rgc_subset = pseudobulks[pseudobulks.obs['cell_type'] == "RGCs"]
pseudobulk_rgc_subset.var_names = data.var_names
inference = DefaultInference(n_cpus=8)
dds_rgc_sub = DeseqDataSet(
    counts=pseudobulk_rgc_subset.X+1,
    metadata=pseudobulk_rgc_subset.obs,
    # design="~ exp_condition + time + exp_condition:time",
    design="~ exp_condition + time",
    refit_cooks=True,
    inference=inference,
)


In [ ]:
dds_rgc_sub.deseq2()
ds_rgc_sub = DeseqStats(dds_rgc_sub, contrast=["exp_condition", "Mtz", "Cntr"], inference=inference)
ds_rgc_sub.summary()

In [ ]:
deseq_volcano(
    ds_rgc_sub,
    pseudobulk_rgc_subset,
    label="RGCs, Ablated vs Unablated \n unadjusted p-values",
    lfc_cutoff=1,pval_cutoff=1.5,
    padj_cutoff=1,
    use_padj=False,
    figsize=(10,7),
    path="./figures/deseq_volcano_rgc_unadjusted.png"
)

In [ ]:
rgc_sub_df = ds_rgc_sub.results_df
rgc_sub_df.index = data.var_names
rgc_sub_df.to_csv("./tables/rgc_sub_deseq.csv")

In [ ]:
np.sum((-1 * np.log10(ds_rgc_sub.padj))>1)

# Muller Glia DE

In [ ]:
pseudobulk_mg_subset = pseudobulks[pseudobulks.obs['cell_type'] == "Muller glia"]
pseudobulk_mg_subset.var_names = data.var_names
inference = DefaultInference(n_cpus=8)
dds_mg_sub = DeseqDataSet(
    counts=pseudobulk_mg_subset.X+1,
    metadata=pseudobulk_mg_subset.obs,
    design="~ exp_condition + time",
    refit_cooks=True,
    inference=inference,
)

dds_mg_sub.deseq2()
ds_mg_sub = DeseqStats(dds_mg_sub, contrast=["exp_condition", "Mtz", "Cntr"], inference=inference)
ds_mg_sub.summary()

In [ ]:
deseq_volcano(
    ds_mg_sub,
    pseudobulk_mg_subset,
    label="Muller glia, Injury vs Control",
    lfc_cutoff=1,pval_cutoff=1,
    padj_cutoff=0.1,
    use_padj=False,
)

In [ ]:
np.sum((-1 * np.log10(ds_mg_sub.padj))>1)

# Signature Sets

In [ ]:
# Selecting signature genes only 

signature = 'parthanatos'
genes_to_mark = data.var[signature]
genes_to_mark = np.array(genes_to_mark)

plt.figure(figsize=(10,10))
plt.title("DESeq2, RGCs only, Change in Injury compared to Control")

x = ds_sub.LFC['exp_condition[T.Mtz]']
y = -np.log10(ds_sub.p_values)
# y = -np.log10(ds_sub.padj)
plt.scatter(
    x,y,
    s=1,
)
ax = plt.gca()
offset = 0.05
for gene,x,y in zip(pseudobulk_subset.var_names[genes_to_mark],x[genes_to_mark],y[genes_to_mark]):
    ax.text(
        x + offset,y + offset,
        gene,fontsize=10)

ylims = ax.get_ylim()
xlims = ax.get_xlim()
plt.plot([lfc_cutoff,lfc_cutoff],ylims,'k--')
plt.plot([-lfc_cutoff,-lfc_cutoff],ylims,'k--')
plt.plot(xlims,[pval_cutoff,pval_cutoff],'k--')

plt.xlabel("Log2 Fold Change")
plt.ylabel("-Log10(p-value)")
plt.tight_layout()
plt.show()

In [ ]:
for signature in ['parthanatos','necroptosis','apoptosis']:
    mask = np.array(data.var[signature])

    print("-"*100)
    print(f"Signature: {signature}")
    print(f"Total: \t{np.sum(ds_sub.LFC['exp_condition[T.Mtz]'][mask])}")
    print(f"\t\t{'\t'.join([f'{g[:6]},' for g in data.var_names[mask]])}")
    print(f"LFCs: \t\t{'\t'.join(np.around(ds_sub.LFC['exp_condition[T.Mtz]'][mask],3).astype(str))}")
    print(f"pvals: \t\t{'\t'.join(np.around(ds_sub.p_values[mask],3).astype(str))}")
    print(f"padjs: \t\t{'\t'.join(np.around(ds_sub.padj[mask],3).astype(str))}")

In [ ]:
diff_sort = np.argsort(ds_sub.LFC['exp_condition[T.Mtz]'])
p_sort = np.argsort(ds_sub.padj)

delta_neg_sub = list(zip(data.var_names[diff_sort],ds_sub.LFC['exp_condition[T.Mtz]'][diff_sort]))[:100]
delta_pos_sub = list(zip(data.var_names[diff_sort],ds_sub.LFC['exp_condition[T.Mtz]'][diff_sort]))[-100:]

# good_p_delta = ds_sub.LFC['exp_condition[T.Mtz]'][p_sort][:100]
# good_p = ds_sub.padj[p_sort][:100]
